# EfficientNetV2-S — Optuna HPO + Ordinal Focal Loss (VRAM-optimized)

Combines:
- **EfficientNetV2-S** backbone (torchvision, ImageNet weights)
- **OrdinalFocalLoss** — BCEWithLogits + focal modulation on ordinal-encoded targets
- **Optuna** — Bayesian search over lr, dropout, gamma, alpha, fine-tune block count, batch size
- **Entropy filtering** — removes top-20% noisiest samples before training

### VRAM optimizations applied
- **AMP (mixed precision)** with `torch.cuda.amp.autocast` + `GradScaler` — ~40% less VRAM
- **Robust cleanup between Optuna trials** via `try/finally` + `gc.collect()` + `empty_cache()`
- **`set_to_none=True`** on `zero_grad` — frees gradient tensors
- **`non_blocking=True`** transfers with `pin_memory=True`
- **`persistent_workers=True`** — avoids recreating workers each epoch
- **Reduced batch size search space** (removed 8) — gradient accumulation available as alternative
- **Peak memory tracking** between trials for diagnostics


In [ ]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import DataLoader
from torch.utils.data.sampler import RandomSampler
from torch.cuda.amp import autocast, GradScaler
from torchvision.models import efficientnet_v2_s, EfficientNet_V2_S_Weights
import albumentations as Albu
from warmup_scheduler import GradualWarmupScheduler
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, f1_score,
    recall_score, precision_score, classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import optuna
from optuna.samplers import TPESampler
import sys
sys.path.append('../../../')
from utils.dataset import PandasDataset


ModuleNotFoundError: No module named 'utils'

## Fixed Configuration

In [ ]:
SEED         = 42
NUM_WORKERS  = 4
OUTPUT_CLASSES = 5      # ordinal thresholds for ISUP 0-5
WARMUP_EPOCHS  = 1
WARMUP_FACTOR  = 2
N_EPOCHS_FULL  = 50     # epochs for the final training run
N_EPOCHS_TRIAL = 8      # epochs per Optuna trial (short proxy)
PATIENCE       = 10     # early stopping patience (full run)
N_TRIALS       = 25     # Optuna trials
VAL_FOLD       = 3      # held-out fold
USE_AMP        = True   # mixed precision (fp16) — major VRAM saver

ROOT_DIR   = '../../..'
DATA_DIR   = '../../../..'
IMAGES_DIR = os.path.join(DATA_DIR, 'tiles')

os.makedirs('logs', exist_ok=True)
os.makedirs('models', exist_ok=True)

MODEL_PATH    = 'models/v2-optuna-ordinal-focal.pth'
LOG_PATH      = 'logs/v2-optuna-ordinal-focal.txt'
OPTUNA_DB     = 'logs/v2-optuna-study.db'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    # cuDNN benchmark speeds things up when input shapes are stable
    torch.backends.cudnn.benchmark = True

print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')


## VRAM Utility Helpers

In [ ]:
def free_vram(*objs):
    """Aggressive cleanup: delete refs, run gc, empty CUDA cache."""
    for o in objs:
        try:
            del o
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def vram_report(tag=''):
    if not torch.cuda.is_available():
        return
    alloc = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f'  [VRAM {tag}] allocated={alloc:.2f}GB reserved={reserved:.2f}GB peak={peak:.2f}GB')


## Loss Function

In [ ]:
class OrdinalFocalLoss(nn.Module):
    """
    Focal loss applied element-wise to ordinal-encoded binary targets.

    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)

    Ordinal encoding:
      ISUP 0 → [0,0,0,0,0]   ISUP 3 → [1,1,1,0,0]
      ISUP 1 → [1,0,0,0,0]   ISUP 4 → [1,1,1,1,0]
      ISUP 2 → [1,1,0,0,0]   ISUP 5 → [1,1,1,1,1]
    """
    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super().__init__()
        self.alpha     = alpha
        self.gamma     = gamma
        self.reduction = reduction

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # cast to fp32 inside loss for numerical stability under AMP
        logits = logits.float()
        targets = targets.float()
        probs    = torch.sigmoid(logits)
        bce      = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t      = probs * targets + (1 - probs) * (1 - targets)
        loss     = self.alpha * (1 - p_t) ** self.gamma * bce

        if self.reduction == 'mean':
            return loss.mean()
        if self.reduction == 'sum':
            return loss.sum()
        return loss


def decode_ordinal_predictions(logits: torch.Tensor) -> torch.Tensor:
    return (torch.sigmoid(logits.float()) > 0.5).sum(dim=1)


## EfficientNetV2-S Wrapper

In [ ]:
class EfficientNetV2Api(nn.Module):
    """
    EfficientNetV2-S wrapper with partial unfreezing.

    Args:
        model:            pretrained EfficientNetV2-S from torchvision
        output_dimensions: number of ordinal thresholds (5 for ISUP 0-5)
        dropout_rate:     dropout before the classification head
        unfreeze_blocks:  how many trailing feature blocks to unfreeze
                          (0 = head only; higher = more backbone unfrozen)
    """
    def __init__(
        self,
        model: nn.Module,
        output_dimensions: int,
        dropout_rate: float = 0.4,
        unfreeze_blocks: int = 2,
    ):
        super().__init__()
        self.model = model

        # Freeze entire backbone
        for param in self.model.parameters():
            param.requires_grad = False

        # Unfreeze trailing feature blocks
        if hasattr(self.model, 'features') and unfreeze_blocks > 0:
            for block in self.model.features[-unfreeze_blocks:]:
                for param in block.parameters():
                    param.requires_grad = True

        # Resolve head input features
        if isinstance(self.model.classifier, nn.Sequential):
            in_features = self.model.classifier[-1].in_features
        else:
            in_features = self.model.classifier.in_features

        self.model.classifier = nn.Identity()

        self.head = nn.Sequential(
            nn.LayerNorm(in_features),
            nn.Dropout(dropout_rate),
            nn.Linear(in_features, output_dimensions),
        )

    def extract(self, x: torch.Tensor) -> torch.Tensor:
        x = self.model(x)
        if x.ndim == 4:
            x = x.mean(dim=[2, 3])
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.extract(x))


## Data Loading with Entropy Filtering

In [ ]:
df_all = pd.read_csv(f'{ROOT_DIR}/data/train_5fold.csv')
df_all.columns = df_all.columns.str.strip()
print(f'Total records: {len(df_all)}')

df_entropy = pd.read_csv(f'{ROOT_DIR}/data/entropy.csv')
df_entropy_sorted = df_entropy.sort_values('difficulty_score', ascending=False)
n_remove = int(len(df_entropy_sorted) * 0.20)
noisy_ids = set(df_entropy_sorted.head(n_remove)['image_id'])
df_all = df_all[~df_all['image_id'].isin(noisy_ids)].reset_index(drop=True)
print(f'After entropy filter: {len(df_all)}')

def drop_missing(df, images_dir):
    exists = df['image_id'].apply(lambda x: os.path.isfile(os.path.join(images_dir, f'{x}.png')))
    return df[exists].reset_index(drop=True)

train_idx = df_all['fold'] != VAL_FOLD
df_train = drop_missing(df_all[train_idx].reset_index(drop=True), IMAGES_DIR)
df_val   = drop_missing(df_all[~train_idx].reset_index(drop=True), IMAGES_DIR)
df_test  = drop_missing(pd.read_csv(f'{ROOT_DIR}/data/test.csv'), IMAGES_DIR)

print(f'Train: {len(df_train)}  Val: {len(df_val)}  Test: {len(df_test)}')
print('Train class distribution:')
print(df_train['isup_grade'].value_counts().sort_index())


## Augmentation

In [ ]:
train_transforms = Albu.Compose([
    Albu.Transpose(p=0.5),
    Albu.VerticalFlip(p=0.5),
    Albu.HorizontalFlip(p=0.5),
    Albu.RandomBrightnessContrast(p=0.3),
    Albu.HueSaturationValue(p=0.2),
])

val_transforms = None


## Training and Validation Helpers (AMP-enabled)

In [ ]:
def make_loaders(batch_size: int):
    train_ds = PandasDataset(IMAGES_DIR, df_train, transforms=train_transforms, format='png')
    val_ds   = PandasDataset(IMAGES_DIR, df_val,   transforms=val_transforms,   format='png')
    test_ds  = PandasDataset(IMAGES_DIR, df_test,  transforms=val_transforms,   format='png')

    common = dict(
        num_workers=NUM_WORKERS,
        pin_memory=True,
        persistent_workers=(NUM_WORKERS > 0),
        prefetch_factor=2 if NUM_WORKERS > 0 else None,
    )

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              sampler=RandomSampler(train_ds), **common)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size,
                              sampler=RandomSampler(val_ds),   **common)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size,
                              shuffle=False, **common)
    return train_loader, val_loader, test_loader


def run_epoch_train(model, loader, optimizer, loss_fn, device, scaler, accum_steps=1):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)

    for step, (imgs, targets, _) in enumerate(tqdm(loader, desc='Train', leave=False)):
        imgs    = imgs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(enabled=USE_AMP, dtype=torch.float16):
            logits = model(imgs)
            loss   = loss_fn(logits, targets) / accum_steps

        scaler.scale(loss).backward()

        if (step + 1) % accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * accum_steps)

    return float(np.mean(losses))


def run_epoch_val(model, loader, loss_fn, device):
    model.eval()
    losses, preds, gts = [], [], []
    with torch.no_grad():
        for imgs, targets, _ in tqdm(loader, desc='Val', leave=False):
            imgs    = imgs.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)
            with autocast(enabled=USE_AMP, dtype=torch.float16):
                logits = model(imgs)
                loss   = loss_fn(logits, targets)
            losses.append(loss.item())
            # immediately move to CPU to free VRAM
            preds.append(decode_ordinal_predictions(logits).cpu())
            gts.append(targets.sum(1).long().cpu())

    preds = torch.cat(preds).numpy()
    gts   = torch.cat(gts).numpy()
    kappa = cohen_kappa_score(gts, preds, weights='quadratic')
    acc   = accuracy_score(gts, preds)
    f1    = f1_score(gts, preds, average='macro', zero_division=0)
    return dict(val_loss=float(np.mean(losses)), val_kappa=kappa, val_acc=acc, val_f1=f1)


## Optuna Objective (with robust VRAM cleanup)

In [ ]:
def build_model(dropout_rate: float, unfreeze_blocks: int) -> nn.Module:
    backbone = efficientnet_v2_s(weights=EfficientNet_V2_S_Weights.DEFAULT)
    return EfficientNetV2Api(
        model=backbone,
        output_dimensions=OUTPUT_CLASSES,
        dropout_rate=dropout_rate,
        unfreeze_blocks=unfreeze_blocks,
    ).to(device)


def objective(trial: optuna.Trial) -> float:
    # Initialize names so the finally block can always del them
    model = optimizer = scheduler = scheduler_cos = scaler = None
    train_loader = val_loader = None

    try:
        # ── Hyperparameter space ────────────────────────────────────────────
        lr              = trial.suggest_float('lr',              1e-5, 5e-3, log=True)
        dropout_rate    = trial.suggest_float('dropout_rate',    0.2,  0.6)
        focal_gamma     = trial.suggest_float('focal_gamma',     0.5,  4.0)
        focal_alpha     = trial.suggest_float('focal_alpha',     0.1,  0.5)
        unfreeze_blocks = trial.suggest_int  ('unfreeze_blocks', 1,    4)
        batch_size      = trial.suggest_categorical('batch_size', [2, 4])  # removed 8
        weight_decay    = trial.suggest_float('weight_decay',    1e-6, 1e-2, log=True)

        # ── Build model, loss, loaders ──────────────────────────────────────
        model   = build_model(dropout_rate, unfreeze_blocks)
        loss_fn = OrdinalFocalLoss(alpha=focal_alpha, gamma=focal_gamma)
        scaler  = GradScaler(enabled=USE_AMP)

        train_loader, val_loader, _ = make_loaders(batch_size)

        optimizer = optim.AdamW(
            [p for p in model.parameters() if p.requires_grad],
            lr=lr / WARMUP_FACTOR,
            weight_decay=weight_decay,
        )
        scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=N_EPOCHS_TRIAL - WARMUP_EPOCHS
        )
        scheduler = GradualWarmupScheduler(
            optimizer, multiplier=WARMUP_FACTOR,
            total_epoch=WARMUP_EPOCHS, after_scheduler=scheduler_cos
        )

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

        best_kappa = -1.0
        for epoch in range(1, N_EPOCHS_TRIAL + 1):
            run_epoch_train(model, train_loader, optimizer, loss_fn, device, scaler)
            metrics = run_epoch_val(model, val_loader, loss_fn, device)
            scheduler.step()

            kappa = metrics['val_kappa']
            trial.report(kappa, epoch)

            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

            if kappa > best_kappa:
                best_kappa = kappa

        vram_report(f'trial {trial.number} end')
        return best_kappa

    finally:
        # Runs even on prune/exception — critical for preventing VRAM leak
        free_vram(model, optimizer, scheduler, scheduler_cos, scaler,
                  train_loader, val_loader)


## Run Optuna Study

In [ ]:
sampler = TPESampler(seed=SEED)
pruner  = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)

study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    pruner=pruner,
    storage=f'sqlite:///{OPTUNA_DB}',
    study_name='v2-ordinal-focal',
    load_if_exists=True,
)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True,
               gc_after_trial=True)  # extra safety: optuna also runs gc

print('\nBest trial:')
best = study.best_trial
print(f'  Value (QWK): {best.value:.4f}')
print('  Params:')
for k, v in best.params.items():
    print(f'    {k}: {v}')


## Optuna Visualizations

In [ ]:
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plt.sca(axes[0])
plot_optimization_history(study)
axes[0].set_title('Optimization History')

plt.sca(axes[1])
plot_param_importances(study)
axes[1].set_title('Hyperparameter Importance')

plt.tight_layout()
plt.savefig('logs/v2-optuna-study.png', dpi=150, bbox_inches='tight')
plt.show()


## Full Training with Best Hyperparameters

In [ ]:
# Final cleanup before the big run
free_vram()

best_params = study.best_params
print('Best hyperparameters:', best_params)

model   = build_model(best_params['dropout_rate'], best_params['unfreeze_blocks'])
loss_fn = OrdinalFocalLoss(alpha=best_params['focal_alpha'], gamma=best_params['focal_gamma'])
scaler  = GradScaler(enabled=USE_AMP)

train_loader, val_loader, test_loader = make_loaders(best_params['batch_size'])

optimizer = optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=best_params['lr'] / WARMUP_FACTOR,
    weight_decay=best_params['weight_decay'],
)
scheduler_cos = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=N_EPOCHS_FULL - WARMUP_EPOCHS
)
scheduler = GradualWarmupScheduler(
    optimizer, multiplier=WARMUP_FACTOR,
    total_epoch=WARMUP_EPOCHS, after_scheduler=scheduler_cos
)

print(f'\nStarting full training for up to {N_EPOCHS_FULL} epochs (patience={PATIENCE})\n')


In [ ]:
history = dict(train_loss=[], val_loss=[], val_kappa=[], val_acc=[], val_f1=[])
best_kappa = 0.0
best_epoch = 0
no_improve = 0

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for epoch in range(1, N_EPOCHS_FULL + 1):
    print(f'Epoch {epoch}/{N_EPOCHS_FULL}')

    train_loss = run_epoch_train(model, train_loader, optimizer, loss_fn, device, scaler)
    metrics    = run_epoch_val(model, val_loader, loss_fn, device)
    scheduler.step()
    lr_now = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    for k in ('val_loss', 'val_kappa', 'val_acc', 'val_f1'):
        history[k].append(metrics[k])

    print(f'  train_loss={train_loss:.5f}  val_loss={metrics["val_loss"]:.5f}  '
          f'kappa={metrics["val_kappa"]:.4f}  acc={metrics["val_acc"]*100:.2f}%  lr={lr_now:.7f}')
    vram_report(f'epoch {epoch}')

    with open(LOG_PATH, 'a') as f:
        f.write(
            f'epoch: {epoch} | lr: {lr_now:.7f} | '
            f'train_loss: {train_loss:.5f} | val_loss: {metrics["val_loss"]:.5f} | '
            f'val_kappa: {metrics["val_kappa"]:.4f} | val_acc: {metrics["val_acc"]:.4f}\n'
        )

    if metrics['val_kappa'] > best_kappa:
        best_kappa = metrics['val_kappa']
        best_epoch = epoch
        no_improve = 0
        torch.save(model.state_dict(), MODEL_PATH)
        print(f'  ** Best model saved (QWK={best_kappa:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch}. Best: epoch {best_epoch} QWK={best_kappa:.4f}')
            break

print(f'\nTraining finished. Best QWK: {best_kappa:.4f} at epoch {best_epoch}')


## Training Curves

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'],   label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(history['val_kappa'], color='orange', label='Val QWK')
axes[0, 1].axvline(best_epoch - 1, color='red', linestyle='--', label=f'Best epoch {best_epoch}')
axes[0, 1].set_title('Quadratic Weighted Kappa')
axes[0, 1].legend(); axes[0, 1].grid(True)

axes[1, 0].plot(history['val_acc'], color='green', label='Val Accuracy')
axes[1, 0].set_title('Accuracy')
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(history['val_f1'], color='red', label='Val Macro F1')
axes[1, 1].set_title('Macro F1')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.suptitle('EfficientNetV2-S — Ordinal Focal Loss (Optuna best params)', y=1.01)
plt.tight_layout()
plt.savefig('logs/v2-optuna-ordinal-focal-training.png', dpi=300, bbox_inches='tight')
plt.show()


## Test-Set Evaluation with Bootstrap CI

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH, weights_only=True))
model.eval()

test_preds, test_gts = [], []
with torch.no_grad():
    for imgs, targets, _ in tqdm(test_loader, desc='Test'):
        imgs = imgs.to(device, non_blocking=True)
        with autocast(enabled=USE_AMP, dtype=torch.float16):
            logits = model(imgs)
        test_preds.append(decode_ordinal_predictions(logits).cpu())
        test_gts.append(targets.sum(1).long())

test_preds = torch.cat(test_preds).numpy()
test_gts   = torch.cat(test_gts).numpy()

# ── Point estimates ───────────────────────────────────────────────────
pt_acc   = accuracy_score(test_gts, test_preds)
pt_kappa = cohen_kappa_score(test_gts, test_preds, weights='quadratic')
pt_f1    = f1_score(test_gts, test_preds, average='macro', zero_division=0)

# ── Bootstrap (1 000 resamples, 95% CI) ──────────────────────────────
N_BOOT = 1000
rng    = np.random.default_rng(SEED)
n      = len(test_gts)
boot   = dict(acc=np.empty(N_BOOT), kappa=np.empty(N_BOOT), f1=np.empty(N_BOOT))

for i in tqdm(range(N_BOOT), desc='Bootstrap'):
    idx            = rng.integers(0, n, size=n)
    boot['acc'][i]   = accuracy_score(test_gts[idx], test_preds[idx])
    boot['kappa'][i] = cohen_kappa_score(test_gts[idx], test_preds[idx], weights='quadratic')
    boot['f1'][i]    = f1_score(test_gts[idx], test_preds[idx], average='macro', zero_division=0)

def ci(arr):
    return arr.std(ddof=1), np.percentile(arr, 2.5), np.percentile(arr, 97.5)

acc_s,   acc_lo,   acc_hi   = ci(boot['acc'])
kap_s,   kap_lo,   kap_hi   = ci(boot['kappa'])
f1_s,    f1_lo,    f1_hi    = ci(boot['f1'])

print('\n' + '='*70)
print('TEST SET RESULTS — EfficientNetV2-S + Ordinal Focal Loss (Optuna)')
print('='*70)
print(f'Accuracy : {pt_acc*100:.2f}% ± {acc_s*100:.2f}%  '
      f'[95% CI: {acc_lo*100:.2f}%–{acc_hi*100:.2f}%]')
print(f'QW Kappa : {pt_kappa:.4f} ± {kap_s:.4f}  '
      f'[95% CI: {kap_lo:.4f}–{kap_hi:.4f}]')
print(f'Macro F1 : {pt_f1:.4f} ± {f1_s:.4f}  '
      f'[95% CI: {f1_lo:.4f}–{f1_hi:.4f}]')
print('='*70)
print()
print(classification_report(test_gts, test_preds,
      target_names=[f'ISUP {i}' for i in range(6)], digits=4, zero_division=0))


## Confusion Matrices

In [ ]:
cm      = confusion_matrix(test_gts, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
labels  = [f'ISUP {i}' for i in range(6)]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0])
axes[0].set_title('Confusion Matrix (counts)')
axes[0].set_ylabel('True'); axes[0].set_xlabel('Predicted')

sns.heatmap(cm_norm, annot=True, fmt='.3f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1])
axes[1].set_title('Confusion Matrix (normalized)')
axes[1].set_ylabel('True'); axes[1].set_xlabel('Predicted')

plt.suptitle('EfficientNetV2-S — Ordinal Focal Loss', y=1.01)
plt.tight_layout()
plt.savefig('logs/v2-optuna-ordinal-focal-confusion-matrix.png', dpi=300, bbox_inches='tight')
plt.show()


## Save Results

In [ ]:
results_path = 'logs/v2-optuna-ordinal-focal-results.txt'
with open(results_path, 'w') as f:
    f.write('EfficientNetV2-S + Ordinal Focal Loss (Optuna HPO)\n')
    f.write('='*70 + '\n\n')
    f.write('Best Optuna hyperparameters:\n')
    for k, v in best_params.items():
        f.write(f'  {k}: {v}\n')
    f.write(f'\nOptuna best val QWK: {study.best_value:.4f}\n\n')
    f.write(f'Bootstrap resamples: {N_BOOT}\n\n')
    f.write(f'Accuracy : {pt_acc*100:.2f}% ± {acc_s*100:.2f}%  '
            f'[95% CI: {acc_lo*100:.2f}%–{acc_hi*100:.2f}%]\n')
    f.write(f'QW Kappa : {pt_kappa:.4f} ± {kap_s:.4f}  '
            f'[95% CI: {kap_lo:.4f}–{kap_hi:.4f}]\n')
    f.write(f'Macro F1 : {pt_f1:.4f} ± {f1_s:.4f}  '
            f'[95% CI: {f1_lo:.4f}–{f1_hi:.4f}]\n\n')
    f.write('Classification Report:\n')
    f.write(classification_report(test_gts, test_preds,
            target_names=labels, digits=4, zero_division=0))
    f.write('\nConfusion Matrix:\n')
    f.write(str(cm) + '\n')

print(f'Results saved to {results_path}')
